# DashLight — Training Notebook
Trains a **YOLO11n** object detection model for dashboard warning lights and exports it to **TFLite** for the SnapMyAuto Flutter app.

**Before you run:**
1. Runtime → Change runtime type → **T4 GPU**
2. Left sidebar → 🔑 **Secrets** → add `ROBOFLOW_API_KEY` (Roboflow → Settings → API Keys) → enable notebook access
3. Fill in the settings in the next cell
4. Runtime → **Run all**

## 1. Settings

In [ ]:
# ---- EDIT THESE ----
RF_WORKSPACE = "snapmyauto"     # Roboflow workspace ID (from project URL)
RF_PROJECT   = "dashlight"      # Roboflow project ID
RF_VERSION   = 1                # dataset version number

MODEL_SIZE = "yolo11n.pt"       # n = nano (fast on phones). Try yolo11s.pt if accuracy is low
EPOCHS     = 100
IMG_SIZE   = 640
BATCH      = 16
PATIENCE   = 20                 # stop early if no improvement for 20 epochs
RUN_NAME   = f"dashlight_v{RF_VERSION}"

## 2. Check GPU

In [ ]:
!nvidia-smi

## 3. Save results to Google Drive
Colab can disconnect. Saving to Drive keeps your training results safe.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/dashlight"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Results will be saved to:", DRIVE_DIR)

## 4. Install libraries

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## 5. Download dataset from Roboflow

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download("yolov8")
DATA_YAML = f"{dataset.location}/data.yaml"
print("Dataset at:", dataset.location)

## 6. Check dataset (images and boxes per class)

In [ ]:
import yaml, glob, collections

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
names = cfg["names"]
print(f"{len(names)} classes:", names)

for split in ["train", "valid", "test"]:
    imgs = glob.glob(f"{dataset.location}/{split}/images/*")
    counts = collections.Counter()
    for lbl in glob.glob(f"{dataset.location}/{split}/labels/*.txt"):
        for line in open(lbl):
            if line.strip():
                counts[int(line.split()[0])] += 1
    print(f"\n{split}: {len(imgs)} images")
    for i, n in enumerate(names):
        flag = "  <-- LOW, add more" if split == "train" and counts[i] < 50 else ""
        print(f"  {n:20s} {counts[i]}{flag}")

## 7. Train

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_SIZE)   # pretrained weights = transfer learning
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    fliplr=0.0,            # no horizontal flip (icons are not symmetric)
    project=f"{DRIVE_DIR}/runs",
    name=RUN_NAME,
    exist_ok=True,
)
RUN_DIR = f"{DRIVE_DIR}/runs/{RUN_NAME}"
BEST = f"{RUN_DIR}/weights/best.pt"
print("Best weights:", BEST)

### Resume if Colab disconnected
Only run this cell if training stopped half-way.

In [ ]:
# from ultralytics import YOLO
# model = YOLO(f"{DRIVE_DIR}/runs/{RUN_NAME}/weights/last.pt")
# model.train(resume=True)

## 8. Evaluate on test set

In [ ]:
best = YOLO(BEST)
metrics = best.val(data=DATA_YAML, split="test", imgsz=IMG_SIZE)

print(f"mAP@50     : {metrics.box.map50:.3f}")
print(f"mAP@50-95  : {metrics.box.map:.3f}")
print(f"Precision  : {metrics.box.mp:.3f}")
print(f"Recall     : {metrics.box.mr:.3f}")

print("\nPer-class mAP@50-95:")
for i, c in enumerate(metrics.box.ap_class_index):
    print(f"  {names[c]:20s} {metrics.box.maps[c]:.3f}")

## 9. Training charts

In [ ]:
from IPython.display import Image, display
for img in ["results.png", "confusion_matrix_normalized.png", "PR_curve.png"]:
    p = f"{RUN_DIR}/{img}"
    if os.path.exists(p):
        print(img); display(Image(filename=p, width=800))

## 10. Test on sample images
Upload your own phone photos to `/content/samples/` (left sidebar → Files) and run.

In [ ]:
os.makedirs("/content/samples", exist_ok=True)
sample_imgs = glob.glob("/content/samples/*") or glob.glob(f"{dataset.location}/test/images/*")[:6]
preds = best.predict(sample_imgs, conf=0.5, save=True, project="/content/preds", name="run", exist_ok=True)
for p in glob.glob("/content/preds/run/*")[:6]:
    display(Image(filename=p, width=500))

## 11. Export to TFLite

In [ ]:
tflite_path = best.export(format="tflite", imgsz=IMG_SIZE)   # float32 + float16
print("Exported:", tflite_path)

# Optional: smaller + faster INT8 version (slight accuracy drop)
# int8_path = best.export(format="tflite", imgsz=IMG_SIZE, int8=True, data=DATA_YAML)

## 12. Package for the app + GitHub

In [ ]:
import shutil, glob, datetime

OUT = "/content/dashlight_export"
os.makedirs(OUT, exist_ok=True)

saved = glob.glob(f"{RUN_DIR}/weights/best_saved_model/*float16.tflite") or [tflite_path]
shutil.copy(saved[0], f"{OUT}/dashlight.tflite")

with open(f"{OUT}/labels.txt", "w") as f:
    f.write("\n".join(names[i] for i in range(len(names))))

for img in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png", "F1_curve.png"]:
    if os.path.exists(f"{RUN_DIR}/{img}"):
        shutil.copy(f"{RUN_DIR}/{img}", OUT)

n_train = len(glob.glob(f"{dataset.location}/train/images/*"))
with open(f"{OUT}/metrics.md", "w") as f:
    f.write(f"# DashLight {RUN_NAME}\n\n")
    f.write(f"- Date: {datetime.date.today()}\n- Base model: {MODEL_SIZE}\n- Epochs: {EPOCHS}, imgsz: {IMG_SIZE}\n")
    f.write(f"- Roboflow: {RF_WORKSPACE}/{RF_PROJECT} v{RF_VERSION}, train images: {n_train}\n\n")
    f.write("| mAP@50 | mAP@50-95 | Precision | Recall |\n|---|---|---|---|\n")
    f.write(f"| {metrics.box.map50:.3f} | {metrics.box.map:.3f} | {metrics.box.mp:.3f} | {metrics.box.mr:.3f} |\n")

shutil.make_archive("/content/dashlight_export", "zip", OUT)
shutil.copy("/content/dashlight_export.zip", DRIVE_DIR)
print(os.listdir(OUT))

from google.colab import files
files.download("/content/dashlight_export.zip")

## Next
- `dashlight.tflite` + `labels.txt` → `snapmyauto-app/assets/models/`
- Charts + `metrics.md` → `dashlight/results/` (commit to GitHub)
- Update the results table in `README.md`